In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.11.0+cu128
CUDA available: True


In [2]:
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4


# Create a Tensor


In [3]:
x = torch.tensor([1,2,3,4]) # tensor lives in cpu

print(x)
print("Shape:", x.shape)
print("Dtype:", x.dtype)
print("Device:", x.device)

tensor([1, 2, 3, 4])
Shape: torch.Size([4])
Dtype: torch.int64
Device: cpu


In [4]:
# move tensor from cpu to gpu
x_gpu = x.to("cuda")
print(x_gpu)

tensor([1, 2, 3, 4], device='cuda:0')


In [5]:
# cpu vs gpu
x = torch.randn(1000, 1000)

x_cpu = x.to("cpu")
x_gpu = x.to("cuda")

print(x_cpu.device)
print(x_gpu.device)

cpu
cuda:0


In [6]:
# a = torch.randn(10)
# b = torch.randn(10).cuda()

# c = a + b # you can not perform addition as both the tensors are from differnet locations


In [7]:
# tensor shapes
x = torch.randn(32, 784)

print(x.shape)
print(x.ndim)
print(x.numel()) # number of elements

torch.Size([32, 784])
2
25088


# data type


In [8]:
x = torch.randn(1000, 1000)

print(x.dtype)
print(x.element_size())

torch.float32
4


# A simple GPU computation


In [9]:
a = torch.randn(20000, 20000, device="cuda")
b = torch.randn(20000, 20000, device="cuda")

c = a @ b

print(c.shape)
print(c.device)
# Python > PyTorch >  a @ b > CUDA operation > GPU kernel > GPU SMs

torch.Size([20000, 20000])
cuda:0


# GPU Synchronization

In [10]:

# CPU launches GPU work; GPU executes it asynchronously; synchronization makes the CPU wait.
a = torch.randn(2000, 2000, device="cuda")
b = torch.randn(2000, 2000, device="cuda")

c = a @ b

torch.cuda.synchronize()

print("GPU computation finished")

# __syncthreads()	--  GPU threads wait for each other / Block-level synchronization
# cudaDeviceSynchronize()	 --  CPU waits for GPU

GPU computation finished


In [11]:
# measuring GPU time
import time

start = time.time()

c = a @ b

end = time.time()

print(end - start)

0.0007638931274414062


In [12]:
# The second synchronization ensures the GPU has finished.
import time

torch.cuda.synchronize()

start = time.time()

c = a @ b

torch.cuda.synchronize()

end = time.time()

print("Time:", end - start)

Time: 0.0034852027893066406


In [13]:
# your first GPU memory Experiment
torch.cuda.empty_cache()

print("Before:",
      torch.cuda.memory_allocated() / 1024**2, "MB")

x = torch.randn(4000, 4000, device="cuda")

print("After:",
      torch.cuda.memory_allocated() / 1024**2, "MB")

Before: 58.45751953125 MB
After: 119.49267578125 MB


In [14]:
# delete the tensor
del x


In [15]:
print("After:",
      torch.cuda.memory_allocated() / 1024**2, "MB")

After: 58.45751953125 MB


# Lets create a  Linear Layer manually and implement forward pass



In [16]:
import torch


batch_size = 32
input_dim = 784
output_dim = 128

X = torch.randn(batch_size, input_dim)

W = torch.randn(output_dim, input_dim)
b = torch.randn(output_dim)

# forward pass
Y = X @ W.T + b

print(Y.shape)

torch.Size([32, 128])


In [17]:
# lets use pytroch Lienar
linear = torch.nn.Linear(input_dim, output_dim)

In [18]:
print(linear.weight.shape)
print(linear.bias.shape)

torch.Size([128, 784])
torch.Size([128])


# parameter
parameter is a tensor that model lerans during training

parameters=Din * ​Dout ​+ Dout​

# Adding non Linearity to the network


In [19]:
# hands on ReLu
import torch

x = torch.tensor([-3.0, -1.0, 0.0, 2.0, 5.0])

y = torch.relu(x)

print("x:", x)
print("y:", y)

# ReLu Creates a mask  that mask comes from the derivative of the ReLu Fucntion = [0, 0, 1, 1, 1, 1]
# So ReLU effectively says:

# Don't send gradient backward through neurons that were inactive.
# pytorch autograd does the same



x: tensor([-3., -1.,  0.,  2.,  5.])
y: tensor([0., 0., 0., 2., 5.])


# hands on cross entropy loss


In [20]:
# cross entropy expects logits
# L=−log(Pcorrect​)


logits = torch.tensor([2.0, 1.0, 0.1])

target = torch.tensor(0)

loss = torch.nn.functional.cross_entropy(
    logits.unsqueeze(0),
    target.unsqueeze(0)
)

print(loss)

tensor(0.4170)


# compare memory before and after the Adam


In [24]:

import torch
import torch.nn as nn
import torch.nn.functional as F


# -----------------------------
# 1. Define the model
# -----------------------------
class MLP(nn.Module):
    def __init__(self, input_dim=784, output_dim=10):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 256)
        self.fc2 = nn.Linear(256, output_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


# -----------------------------
# 2. Create fixed input data
# -----------------------------
X = torch.randn(32, 784, device="cuda")
target = torch.randint(0, 10, (32,), device="cuda")


# -----------------------------
# 3. Function to test optimizer
# -----------------------------
def test_optimizer(optimizer_name):

    # Create a fresh model
    model = MLP().cuda()

    # Create optimizer
    if optimizer_name == "Adam":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=0.001
        )

    elif optimizer_name == "SGD":
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=0.001
        )

    # Make sure previous CUDA work is finished
    torch.cuda.synchronize()

    # Reset CUDA memory statistics
    torch.cuda.reset_peak_memory_stats()

    # Memory before training
    before = torch.cuda.memory_allocated()

    # -----------------------------
    # Forward + backward
    # -----------------------------
    optimizer.zero_grad()

    logits = model(X)

    loss = F.cross_entropy(logits, target)

    loss.backward()

    torch.cuda.synchronize()

    after_backward = torch.cuda.memory_allocated()

    # -----------------------------
    # Optimizer step
    # -----------------------------
    optimizer.step()

    torch.cuda.synchronize()

    after_step = torch.cuda.memory_allocated()

    # Peak memory
    peak = torch.cuda.max_memory_allocated()

    # -----------------------------
    # Print results
    # -----------------------------
    print(f"\n===== {optimizer_name} =====")

    print(
        f"Before:          {before / 1024**2:.2f} MB"
    )

    print(
        f"After backward:  {after_backward / 1024**2:.2f} MB"
    )

    print(
        f"After step:      {after_step / 1024**2:.2f} MB"
    )

    print(
        f"Peak memory:     {peak / 1024**2:.2f} MB"
    )

    # Count optimizer state
    optimizer_state_elements = 0

    for state in optimizer.state.values():
        for value in state.values():
            if torch.is_tensor(value):
                optimizer_state_elements += value.numel()

    print(
        f"Optimizer state: {optimizer_state_elements:,} elements"
    )


# -----------------------------
# 4. Run both experiments
# -----------------------------

test_optimizer("SGD")

test_optimizer("Adam")


===== SGD =====
Before:          57.96 MB
After backward:  58.73 MB
After step:      58.73 MB
Peak memory:     58.77 MB
Optimizer state: 0 elements

===== Adam =====
Before:          57.96 MB
After backward:  58.73 MB
After step:      60.29 MB
Peak memory:     61.06 MB
Optimizer state: 407,064 elements


# GPU Profiling with torch.profiler

In [26]:
# create MLP

import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda"


class MLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.linear1 = nn.Linear(784, 128)
        self.linear2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.linear1(x)
        x = F.relu(x)
        x = self.linear2(x)
        return x


model = MLP().to(device)

In [27]:
# input

X = torch.randn(32, 784, device=device)
target = torch.randint(0, 10, (32,), device=device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01
)

In [28]:
with torch.profiler.profile(
    activities=[
        torch.profiler.ProfilerActivity.CPU,
        torch.profiler.ProfilerActivity.CUDA,
    ],
    record_shapes=True,
    profile_memory=True
) as prof:

    optimizer.zero_grad()

    with torch.profiler.record_function("forward"):
        logits = model(X)

    with torch.profiler.record_function("loss"):
        loss = criterion(logits, target)

    with torch.profiler.record_function("backward"):
        loss.backward()

    with torch.profiler.record_function("optimizer"):
        optimizer.step()


torch.cuda.synchronize()

/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


In [29]:
print(
    prof.key_averages().table(
        sort_by="cuda_time_total",
        row_limit=30
    )
)

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                forward         0.00%       0.000us         0.00%       0.000us       0.000us     410.237us       251.87%     410.237us     410.237us           0 B           0 B           0 B           0 